In [ ]:
import warnings
import pandas as pd
import os


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Function definitions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    # Apply entry time offset to the signal time
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration', 'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        signal_open_price = price_data.at[signal_datetime + pd.Timedelta(minutes=entry_time_offset) , 'Open']
        
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if result == 1:
            current_margin = current_margin * (1 + tp)
        elif result == -1:
            current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

In [2]:
import numpy as np
from itertools import product
from datetime import datetime
import pandas as pd 

def optimize_parameters(price_data, signal_data, month, initial_margin=100000):
    # Define start and end date for the given month
    start_date = datetime(month.year, month.month, 1)
    if month.month == 12:
        end_date = datetime(month.year + 1, 1, 1)
    else:
        end_date = datetime(month.year, month.month + 1, 1)
    
    # Log the date range for filtering
    print(f"Filtering data from {start_date} to {end_date}")

    # Ensure the price_data and signal_data are properly indexed by datetime
    if not isinstance(price_data.index, pd.DatetimeIndex):
        print("Converting price_data index to DatetimeIndex")
        price_data.index = pd.to_datetime(price_data.index)
    
    if 'Datetime' not in signal_data.columns:
        print("Datetime column not found in signal_data")
        return None

    # Convert signal_data 'Datetime' column to datetime if not already
    if not np.issubdtype(signal_data['Datetime'].dtype, np.datetime64):
        print("Converting signal_data 'Datetime' column to datetime")
        signal_data['Datetime'] = pd.to_datetime(signal_data['Datetime'])
    
    # Filter data for the given month
    month_price_data = price_data[start_date:end_date]
    month_signal_data = signal_data[(signal_data['Datetime'] >= start_date) & (signal_data['Datetime'] < end_date)]
    
    # Log the filtered data to ensure it is correct
    print(f"Filtered price data for the month:\n{month_price_data}")
    print(f"Filtered signal data for the month:\n{month_signal_data}")
    
    if month_price_data.empty or month_signal_data.empty:
        print("Filtered data is empty, returning early.")
        return None
    
    # Define the parameter space with correct ranges
    tp_values = np.arange(0.0005, 0.02, 0.0001)
    sl_values = np.arange(0.0005, 0.02, 0.0001)
    entry_time_offset_values = np.arange(0, 240, 1)
    percentage_change_values = np.arange(0, 0.01, 0.0001)
    
    best_roi = -np.inf
    best_params = None
    
    for tp, sl, entry_time_offset, percentage_change in product(tp_values, sl_values, entry_time_offset_values, percentage_change_values):
        print(f"Testing combination: TP={tp}, SL={sl}, Entry Time Offset={entry_time_offset}, Percentage Change={percentage_change}")
        result = backtest_trades(
            month_price_data,
            month_signal_data,
            tp=tp,
            sl=sl,
            entry_time_offset=entry_time_offset,
            percentage_change=percentage_change,
            time_limit_minutes=120
        )
        if result.empty:
            print("Result is empty, skipping...")
            continue
        final_nav = result.iloc[-1]['NAV']
        roi = ((final_nav - initial_margin) / initial_margin)*100
        print(f"Resulting ROI: {roi}")
        if roi > best_roi:
            best_roi = roi
            best_params = (tp, sl, entry_time_offset, percentage_change)
    
    if best_params is None:
        print("No valid parameter combinations found.")
        return None
    
    optimized_params = {
        'tp': best_params[0],
        'sl': best_params[1],
        'entry_time_offset': best_params[2],
        'percentage_change': best_params[3],
        'max_roi': best_roi
    }
    
    return optimized_params

# Load the new datasets
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'], index_col='Datetime')
signal_data = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv', parse_dates=['Datetime'])

# Log the first few rows of the datasets to ensure they are loaded correctly
print(f"Loaded price data:\n{price_data.head()}")
print(f"Loaded signal data:\n{signal_data.head()}")

# Example usage for January 2024
month = datetime(2024, 1, 1)
optimized_params = optimize_parameters(price_data, signal_data, month)

if optimized_params:
    optimized_params
else:
    print("No optimized parameters found.")


Loaded price data:
                        Open     High      Low    Close   volume
Datetime                                                        
2024-01-01 00:00:00  42314.0  42335.8  42289.6  42331.9  289.641
2024-01-01 00:01:00  42331.9  42353.1  42331.8  42350.4  202.444
2024-01-01 00:02:00  42350.4  42370.8  42349.6  42360.2  271.521
2024-01-01 00:03:00  42360.1  42405.8  42360.1  42405.8  392.238
2024-01-01 00:04:00  42405.7  42437.2  42405.7  42437.1  568.366
Loaded signal data:
             Datetime  Signal
0 2024-01-01 00:00:00      -2
1 2024-01-01 16:00:00       2
2 2024-01-02 00:00:00      -2
3 2024-01-02 22:00:00       2
4 2024-01-03 16:00:00       2
Filtering data from 2024-01-01 00:00:00 to 2024-02-01 00:00:00
Filtered price data for the month:
                        Open     High      Low    Close   volume
Datetime                                                        
2024-01-01 00:00:00  42314.0  42335.8  42289.6  42331.9  289.641
2024-01-01 00:01:00  42331.9  423

KeyboardInterrupt: 